# Checkpoint Use-Case Recipes

Short, copy-paste patterns for the new checkpoint/lineage API.


## Setup


In [ ]:
from pathlib import Path

from hypergraph import Graph, SyncRunner, node
from hypergraph.checkpointers import SqliteCheckpointer
from hypergraph.exceptions import WorkflowAlreadyCompletedError

DB_PATH = Path("./tmp/checkpoint-recipes.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
for p in [DB_PATH, Path(str(DB_PATH) + "-shm"), Path(str(DB_PATH) + "-wal")]:
    p.unlink(missing_ok=True)

cp = SqliteCheckpointer(str(DB_PATH))
runner = SyncRunner(checkpointer=cp)


@node(output_name="doubled")
def double(x: int) -> int:
    return x * 2


graph = Graph([double])

fail_switch = {"on": True}


@node(output_name="seed")
def seed(x: int) -> int:
    return x


@node(output_name="out")
def maybe_fail(seed: int) -> int:
    if fail_switch["on"]:
        raise RuntimeError("transient")
    return seed * 10


flaky_graph = Graph([seed, maybe_fail])

## Recipe 1: Start a new workflow (auto ID)


In [ ]:
result = runner.run(graph, {"x": 5})
print("auto workflow id:", result.workflow_id)

## Recipe 2: Same `workflow_id` is strict

For a completed workflow, same-ID rerun is rejected.


In [ ]:
runner.run(graph, {"x": 5}, workflow_id="job-1")

try:
    runner.run(graph, workflow_id="job-1")
except WorkflowAlreadyCompletedError as e:
    print(type(e).__name__)

## Recipe 3: Override values by forking


In [ ]:
branch_cp = cp.checkpoint("job-1")
forked = runner.run(graph, {"x": 100}, checkpoint=branch_cp, workflow_id="job-1-fork")
print("fork:", forked.workflow_id, "doubled=", forked["doubled"])

## Recipe 3b: One-line override (auto-fork)

When a workflow id already exists, this is the ergonomic shortcut.


In [ ]:
overridden = runner.run(
    graph,
    {"x": 200},
    workflow_id="job-1",
    override_workflow=True,
)
print("new forked workflow_id:", overridden.workflow_id)
print("doubled:", overridden["doubled"])

## Recipe 4: Resume a failed workflow (implicit retry)


In [ ]:
first = runner.run(flaky_graph, {"x": 7}, workflow_id="job-failed", error_handling="continue")
print("first status:", first.status.value)

# Fix the transient issue, then resume same workflow_id.
fail_switch["on"] = False
resumed = runner.run(flaky_graph, workflow_id="job-failed")
print("resumed id:", resumed.workflow_id, "out=", resumed["out"])

## Recipe 5: Ask for checkpoint/state by workflow ID


In [ ]:
checkpoint = cp.checkpoint("job-1")
state = cp.state("job-1")
steps = cp.steps("job-1")
print("state keys:", sorted(state.keys()))
print("step count:", len(steps))

## Recipe 6: Inspect git-like lineage lanes


In [ ]:
lineage = cp.lineage("job-1-fork")
lineage

## Recipe 7: Hashes for compatibility vs caching


In [ ]:
print("structural hash:", graph.structural_hash[:12])
print("code hash:", graph.code_hash[:12])

In [ ]:
# Optional cleanup
for p in [DB_PATH, Path(str(DB_PATH) + "-shm"), Path(str(DB_PATH) + "-wal")]:
    p.unlink(missing_ok=True)

In [ ]:
import contextlib

# Close checkpointer connection (safe to run multiple times)
with contextlib.suppress(Exception):
    await cp.close()